# Day 11: Word Embeddings

**Goal:** Replace one-hot vectors with dense, meaningful representations of words.

### The problem we're solving

Yesterday's BoW classifier treats every word as completely unrelated to every other:
- "good" and "great" → as different as "good" and "banana"
- New word? → just `<unk>`, no semantic info

### Today's solution

**Embeddings** = learned dense vectors that capture meaning.

```
One-hot:  "good"  = [0, 0, 0, ..., 1, 0, 0, ...]   (length 5000, mostly zeros)
                    
Embedding: "good"  = [0.21, -0.45, 0.83, ..., 0.12]  (length 100, all meaningful)
           "great" = [0.19, -0.42, 0.85, ..., 0.10]  ← VERY SIMILAR vector!
           "bad"   = [-0.3, 0.55, -0.2, ..., -0.4]   ← OPPOSITE
```

Today we'll:
1. See WHY one-hot is wasteful
2. Use `nn.Embedding` to learn dense vectors
3. Measure word similarity with cosine
4. Visualize the learned embedding space
5. Build a sentiment classifier using embeddings (improves on Day 10!)
6. Demo word analogies (king - man + woman = queen)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
import re
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Why One-Hot is Wasteful

Let's compare the storage and similarity properties of one-hot vs embeddings:

In [ ]:
# Imagine a vocab of 50,000 words

vocab_size = 50_000

# One-hot for "good" (let's say index 5)
one_hot_good = torch.zeros(vocab_size)
one_hot_good[5] = 1

# One-hot for "great" (index 8)
one_hot_great = torch.zeros(vocab_size)
one_hot_great[8] = 1

# Storage size comparison
embedding_dim = 100
one_hot_size = vocab_size * 4    # 4 bytes per float
embedding_size = embedding_dim * 4

print(f"VOCAB SIZE: {vocab_size}")
print(f"\nStorage per word:")
print(f"  One-hot (length {vocab_size}): {one_hot_size:,} bytes")
print(f"  Embedding (length {embedding_dim}): {embedding_size:,} bytes")
print(f"  Embedding is {one_hot_size // embedding_size}x smaller!")

# Similarity comparison
def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm() + 1e-8)

print(f"\nSimilarity between 'good' and 'great' using one-hot:")
print(f"  cosine = {cosine_sim(one_hot_good, one_hot_great):.4f}")
print(f"  → No similarity at all! (one-hot vectors are orthogonal)")
print(f"\nThis is the fundamental flaw with one-hot encoding.")

## 2. `nn.Embedding` — A Learnable Lookup Table

`nn.Embedding(vocab_size, dim)` creates a `(vocab_size × dim)` matrix of learnable weights. Each row is the embedding for that word index.

When you call `embedding(token_ids)`, it just looks up those rows — that's it.

In [ ]:
# Create an embedding layer

vocab_size = 10        # tiny vocab for demo
dim = 4                # tiny embedding for visibility

embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=dim)

# The internal weight matrix
print(f"Embedding weight shape: {embedding.weight.shape}")
print(f"  → {vocab_size} rows × {dim} cols (one row per word)")
print(f"\nThe full embedding table:")
print(embedding.weight.data)

# Look up specific words
word_ids = torch.tensor([2, 5, 0, 9])     # 4 words
vectors = embedding(word_ids)

print(f"\nLooking up word IDs {word_ids.tolist()}:")
print(f"Result shape: {vectors.shape}  ({len(word_ids)} words × {dim} dim each)")
print(f"Vectors:\n{vectors.data}")
print(f"\nNotice: row 2 of the table = the vector for word ID 2. It's just a lookup!")

## 3. Sentiment Classifier with Embeddings

Now let's improve on Day 10's BoW classifier. New pipeline:

```
text → tokens → token IDs → Embedding lookup → mean → Linear → prediction
                              ↑
                       The new piece
```

Each word becomes a dense vector. We average them to get a sentence vector. Then classify.

In [ ]:
# Same dataset as Day 10 (movie reviews)

reviews = [
    ("This movie was amazing! Best film I've seen all year.", 1),
    ("Wonderful acting and a beautiful story. Highly recommended.", 1),
    ("Loved every minute. The cinematography was stunning.", 1),
    ("Fantastic performances by the lead actors. Brilliant film.", 1),
    ("An incredible journey. The ending was perfect.", 1),
    ("Heartwarming and inspiring. A must-watch masterpiece.", 1),
    ("Hilarious from start to finish. Great writing and direction.", 1),
    ("The best film of the decade. Stunning visuals and great plot.", 1),
    ("Truly excellent. The acting was superb and the music wonderful.", 1),
    ("I absolutely loved this movie. So engaging and beautiful.", 1),
    ("Perfectly paced and brilliantly acted. A real gem.", 1),
    ("Outstanding work. The director did a fantastic job.", 1),
    ("This film is a masterpiece. Every scene is gorgeous.", 1),
    ("Phenomenal acting and a captivating story throughout.", 1),
    ("A beautiful and moving film. Truly inspiring.", 1),
    ("Terrible movie. Complete waste of time.", 0),
    ("Boring and predictable. The acting was awful.", 0),
    ("I hated every minute. Worst film I've ever seen.", 0),
    ("Awful direction and weak script. Skip this one.", 0),
    ("Painfully slow. The plot made no sense at all.", 0),
    ("Terrible acting and a stupid story. Avoid this film.", 0),
    ("So boring. I almost fell asleep watching it.", 0),
    ("A complete disaster. The worst movie of the year.", 0),
    ("Horrible writing. The actors looked confused throughout.", 0),
    ("Disappointing and dull. Not worth your time.", 0),
    ("I want my money back. This was a terrible experience.", 0),
    ("Cliched and badly acted. Skip this awful film.", 0),
    ("Truly bad. The plot was confusing and the acting weak.", 0),
    ("Painful to watch. Terrible direction and worse acting.", 0),
    ("Boring, predictable, awful. Do not waste your time.", 0),
]

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9' ]+", " ", text)
    return text.split()

# Build vocabulary
np.random.seed(0)
indices = np.random.permutation(len(reviews))
split = int(0.8 * len(reviews))
train_data = [reviews[i] for i in indices[:split]]
test_data = [reviews[i] for i in indices[split:]]

word_counts = Counter()
for text, _ in train_data:
    word_counts.update(tokenize(text))

vocab = ['<unk>', '<pad>'] + [w for w, _ in word_counts.most_common()]
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

print(f"Vocab size: {vocab_size}")
print(f"Train: {len(train_data)} samples")
print(f"Test:  {len(test_data)} samples")

In [ ]:
# Convert text to TOKEN ID lists (not counts!)
# Different from BoW — we keep the sequence

PAD_IDX = word_to_idx['<pad>']

def text_to_ids(text, max_len=20):
    """Convert text to a list of token IDs, padded/truncated to max_len."""
    ids = [word_to_idx.get(t, 0) for t in tokenize(text)]
    # Pad if too short, truncate if too long
    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return ids

# Convert all data
X_train = torch.tensor([text_to_ids(t) for t, _ in train_data], dtype=torch.long)
y_train = torch.tensor([l for _, l in train_data], dtype=torch.long)
X_test  = torch.tensor([text_to_ids(t) for t, _ in test_data],  dtype=torch.long)
y_test  = torch.tensor([l for _, l in test_data],  dtype=torch.long)

print(f"X_train shape: {X_train.shape}  (samples × max_len)")
print(f"y_train shape: {y_train.shape}")
print(f"\nFirst sample:")
print(f"  Text:    '{train_data[0][0]}'")
print(f"  IDs:     {X_train[0].tolist()}")
print(f"  Decoded: {[idx_to_word[i.item()] for i in X_train[0]]}")

In [ ]:
# Build the embedding-based classifier

class EmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, num_classes=2, pad_idx=1):
        super().__init__()
        # Embedding layer: vocab_size words → embed_dim vectors
        # padding_idx: tells the layer to keep <pad> at exactly zero
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        # x shape: (batch, seq_len) — token IDs
        embedded = self.embedding(x)             # (batch, seq_len, embed_dim)
        # Average across sequence dim — sentence vector
        sentence_vec = embedded.mean(dim=1)      # (batch, embed_dim)
        return self.fc(sentence_vec)             # (batch, num_classes)

# Build it
torch.manual_seed(42)
model = EmbeddingClassifier(vocab_size, embed_dim=16, pad_idx=PAD_IDX)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Embedding matrix: {vocab_size} × 16 = {vocab_size * 16:,}")
print(f"  Classifier:       16 × 2 + 2 = 34")

In [ ]:
# Train it

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()

train_losses, test_accs = [], []

for epoch in range(200):
    model.train()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    
    model.eval()
    with torch.no_grad():
        preds = model(X_test).argmax(dim=1)
        acc = (preds == y_test).float().mean().item()
    test_accs.append(acc)
    
    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d}: loss={loss.item():.4f}, test_acc={acc*100:.0f}%")

print(f"\nFinal test accuracy: {test_accs[-1]*100:.0f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(train_losses, 'b-', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training loss'); ax1.grid(True, alpha=0.3)

ax2.plot([a*100 for a in test_accs], 'g-', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Test accuracy'); ax2.set_ylim(0, 105); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Inspect What the Embeddings Learned

The model learned 16-dim vectors for every word. Words that mean similar things should have similar vectors. Let's check:

In [ ]:
# Find similar words using cosine similarity

emb_matrix = model.embedding.weight.data       # (vocab_size, embed_dim)

def cosine_sim_matrix(query, all_vecs):
    """Cosine similarity between one vector and all rows of a matrix."""
    query_norm = query / (query.norm() + 1e-8)
    all_norm = all_vecs / (all_vecs.norm(dim=1, keepdim=True) + 1e-8)
    return all_norm @ query_norm

def most_similar(word, top_k=5):
    if word not in word_to_idx:
        return f"'{word}' not in vocab"
    word_idx = word_to_idx[word]
    query = emb_matrix[word_idx]
    sims = cosine_sim_matrix(query, emb_matrix)
    
    # Mask out the word itself + special tokens
    sims[word_idx] = -2
    sims[0] = -2  # <unk>
    sims[1] = -2  # <pad>
    
    top = sims.topk(top_k)
    return [(idx_to_word[i.item()], s.item()) for s, i in zip(top.values, top.indices)]

# Check similarity for sentiment-laden words
test_words = ['great', 'terrible', 'good', 'bad', 'amazing', 'awful']
print("Most similar words (according to learned embeddings):\n")
for word in test_words:
    sims = most_similar(word)
    if isinstance(sims, str):
        print(f"  {word}: {sims}")
    else:
        print(f"  {word:>10} → ", end="")
        print(", ".join([f"{w} ({s:.2f})" for w, s in sims]))

### Notice what happened:

The model learned that "great" is close to other positive words ("amazing", "wonderful", etc.) — even though we never told it that explicitly!

It learned this because positive words appear in similar contexts in our training data. The training pressure of the sentiment task naturally clusters them together.

**This is the core insight of word embeddings.** With more data (millions of documents), embeddings get even more accurate.

---

## 5. Visualize the Embedding Space

We have 16-dim vectors — too many to plot. Let's project to 2D using PCA:

In [ ]:
# Visualize embeddings in 2D using simple PCA (no sklearn needed)

def pca_2d(matrix):
    """Project a matrix to 2D using PCA (with NumPy)."""
    centered = matrix - matrix.mean(axis=0, keepdims=True)
    # Compute SVD
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    # First 2 components
    return centered @ Vt[:2].T

# Get embeddings for sentiment-laden words
positive_words = ['great', 'amazing', 'wonderful', 'beautiful', 'fantastic', 
                   'brilliant', 'perfect', 'best', 'loved', 'masterpiece']
negative_words = ['terrible', 'awful', 'boring', 'worst', 'bad',
                   'horrible', 'hated', 'disappointing', 'painful', 'confusing']

# Filter to words actually in vocab
def safe_get(words):
    return [w for w in words if w in word_to_idx]

pos_w = safe_get(positive_words)
neg_w = safe_get(negative_words)

# Get the embeddings
pos_idx = [word_to_idx[w] for w in pos_w]
neg_idx = [word_to_idx[w] for w in neg_w]
all_words = pos_w + neg_w
all_idx = pos_idx + neg_idx
all_embeds = emb_matrix[all_idx].numpy()

# Project to 2D
projected = pca_2d(all_embeds)

# Plot
plt.figure(figsize=(10, 7))
n_pos = len(pos_w)

plt.scatter(projected[:n_pos, 0], projected[:n_pos, 1], 
            c='green', s=200, alpha=0.6, edgecolor='black', label='Positive')
plt.scatter(projected[n_pos:, 0], projected[n_pos:, 1], 
            c='red', s=200, alpha=0.6, edgecolor='black', label='Negative')

for i, word in enumerate(all_words):
    plt.annotate(word, (projected[i, 0], projected[i, 1]), 
                 fontsize=11, ha='center', va='center', fontweight='bold')

plt.title('Word embeddings projected to 2D (PCA)')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Positive words cluster on one side, negative on the other!")
print("The model learned 'sentiment' as a direction in the embedding space.")

## 6. Word Analogies — The Famous Demo

The classic Word2Vec result: `king - man + woman ≈ queen`. We don't have those words in our small vocab, but we can demonstrate the principle:

In [ ]:
# Demonstrate vector arithmetic
# Our embeddings are tiny + trained on tiny data, so analogies won't be amazing
# But the math itself is what matters

def analogy(positive, negative, top_k=5):
    """Find the word closest to: positive[0] + positive[1] - negative[0]"""
    valid_pos = [w for w in positive if w in word_to_idx]
    valid_neg = [w for w in negative if w in word_to_idx]
    if len(valid_pos) != len(positive) or len(valid_neg) != len(negative):
        return f"Some words not in vocab"
    
    # Sum positive embeddings, subtract negative embeddings
    target = sum(emb_matrix[word_to_idx[w]] for w in positive)
    target = target - sum(emb_matrix[word_to_idx[w]] for w in negative)
    
    # Find most similar word
    sims = cosine_sim_matrix(target, emb_matrix)
    
    # Mask the input words and special tokens
    for w in positive + negative:
        sims[word_to_idx[w]] = -2
    sims[0] = -2; sims[1] = -2
    
    top = sims.topk(top_k)
    return [(idx_to_word[i.item()], s.item()) for s, i in zip(top.values, top.indices)]

# Demonstrate the principle: even with our tiny model, we can do vector arithmetic
print("VECTOR ARITHMETIC ON EMBEDDINGS:\n")

# This is more about showing the MECHANISM than getting perfect results
# (our 30-review dataset is way too small for great analogies)

queries = [
    (['amazing'], ['terrible']),       # what direction does pos-vs-neg go?
    (['boring'], ['hilarious']),
    (['awful'], ['wonderful']),
]

for pos, neg in queries:
    result = analogy(pos, neg)
    if isinstance(result, str):
        print(f"  {' '.join('+'+w for w in pos)} {' '.join('-'+w for w in neg)} = {result}")
    else:
        words = ', '.join([f"{w} ({s:.2f})" for w, s in result[:3]])
        print(f"  +{pos[0]} -{neg[0]} → {words}")

print("\nNote: real Word2Vec embeddings (trained on billions of words)")
print("show stunning analogies like 'king - man + woman = queen'.")
print("Our 30-review dataset is too small for that level — but the MECHANISM is the same.")

## 7. Comparing Approaches: BoW vs Embeddings

Side-by-side, what's better?

In [ ]:
# Test both approaches on new reviews

def predict_with_embedding(text, model):
    """Predict using the embedding model."""
    ids = torch.tensor([text_to_ids(text)], dtype=torch.long)
    model.eval()
    with torch.no_grad():
        logits = model(ids)
        probs = F.softmax(logits, dim=1)
    pred_class = probs.argmax(dim=1).item()
    confidence = probs[0, pred_class].item()
    return ('positive' if pred_class == 1 else 'negative'), confidence

new_reviews = [
    "What an amazing film! Beautiful and inspiring.",
    "Terrible. Awful. Truly bad.",
    "I loved the acting. Great movie!",
    "Painfully boring. Skip this one.",
    "The film was wonderful. The acting was superb.",
    "Such a bad film. Worst I've seen.",
]

print("Embedding classifier predictions:\n")
print(f"{'Prediction':>10} | {'Confidence':>10} | Review")
print("-" * 80)
for review in new_reviews:
    label, conf = predict_with_embedding(review, model)
    print(f"  {label:>8} | {conf*100:>9.1f}% | '{review}'")

---

## Exercises

1. **Bigger embedding dim:** Set `embed_dim=64` instead of 16. Does similarity quality improve? Are similar words more clustered?

2. **Visualize before vs after training:** Get the embedding matrix BEFORE training (random init) and AFTER training. Plot both with PCA. The trained one should show structure!

3. **Sum vs mean:** Replace `embedded.mean(dim=1)` with `embedded.sum(dim=1)`. Does it still work? Why might mean be better for variable-length sentences?

4. **Pre-trained embeddings:** Try loading GloVe embeddings (available via `gensim` library or torchtext). Use them frozen vs fine-tuned.

5. **Visualize the SENTENCE embeddings:** Compute the mean embedding for each training review, project to 2D, color by class. Are positive and negative reviews separated?

---

## Key Takeaways

### Embedding vs One-hot

| Property | One-hot | Embedding |
|----------|---------|-----------|
| Length | vocab_size (e.g., 50,000) | small (e.g., 100) |
| Density | sparse (1 nonzero) | dense |
| Captures meaning? | No | Yes (after training) |
| Similar words? | All equally distant | Similar words → close |
| Storage | Wasteful | Efficient |
| Trainable? | No | Yes |

### The pipeline now

```
text                          (raw)
   ↓ tokenize
['this', 'movie', 'rocks']    (tokens)
   ↓ word_to_idx
[3, 14, 99]                   (token IDs)
   ↓ nn.Embedding(vocab, dim) ← THE LEARNABLE LOOKUP TABLE
[[v_this], [v_movie], [v_rocks]]   (embedded — shape: seq × dim)
   ↓ mean across sequence
[sentence_vec]                (shape: dim)
   ↓ Linear classifier
[logits]                       (shape: num_classes)
```

### Why this is everywhere

Every modern NLP model — GPT, BERT, Claude, Llama — starts the same way:

```python
self.token_embedding = nn.Embedding(vocab_size, embed_dim)
```

You just learned the FIRST layer of an LLM!

### Limitations still:

- **Order ignored** (mean discards it) → fixed by sequence models
- **No context** ("bank" of a river vs "bank" for money) → fixed by transformers (Day 19+)
- **Tiny vocab handles unknown words poorly** → fixed by subword tokenization (BPE, Day 21)

**Tomorrow:** Day 12 — a complete sentiment classifier project bringing together everything from Days 8-11.